# Hotel cancellation model: training pipeline

Predicts whether a hotel booking will be canceled, so a revenue manager can
decide how many extra rooms to sell (overbooking). The same logistic
regression is trained three ways: scikit-learn, a manual PyTorch loop, and
the standard `torch.nn.Module` + `torch.optim` workflow.

Run start to finish with `uv run python main.py train`.

## 1. Load the data

The CSV is a committed copy of the Kaggle dataset (see
`scripts/fetch_data.py`), so this runs offline and without Kaggle
credentials.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pipeline import (
    BINARY_FEATURES,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    add_features,
    clean,
)

DATA_PATH = Path("data/hotel_reservations.csv")

raw = pd.read_csv(DATA_PATH)
print("Raw shape:", raw.shape)

## 2. Clean

`clean` lives in `pipeline.py` because the app must apply exactly the same
fixes to new bookings. It fixes the invalid 29 Feb 2018 dates, adds quality
flags, caps implausible child counts, and builds the target `y`
(1 = canceled).

In [ ]:
df = clean(raw)

print("Rows moved from 29 Feb to 1 Mar:", df["invalid_date"].sum())
print("Zero-night bookings:            ", df["zero_nights"].sum())
print("Free rooms:                     ", df["is_free"].sum())
print("Max children after capping:     ", df["no_of_children"].max())
print("Missing arrival dates:          ", df["arrival"].isna().sum())
print("Overall cancellation rate:      ", round(df["y"].mean(), 3))

## 3. Time-based split

The model will be trained on past bookings and used on future ones, so the
split follows time: a random split would let the model see later bookings
during training and make the test score look better than it would be in
real use. Cancellation rates also change a lot over time, which a random
split would hide.

- Train: arrivals up to 30 Jun 2018 (fit the models)
- Validation: Jul to Sep 2018 (choose learning rate, regularization, threshold)
- Test: Oct to Dec 2018 (final score, used once)

In [ ]:
VAL_START = pd.Timestamp("2018-07-01")
TEST_START = pd.Timestamp("2018-10-01")

train_df = df[df["arrival"] < VAL_START]
val_df = df[(df["arrival"] >= VAL_START) & (df["arrival"] < TEST_START)]
test_df = df[df["arrival"] >= TEST_START]

assert len(train_df) + len(val_df) + len(test_df) == len(df)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(
        f"{name:<11} {part['arrival'].min():%Y-%m-%d} to {part['arrival'].max():%Y-%m-%d}"
        f"   rows = {len(part):>6,}   cancel rate = {part['y'].mean():.3f}"
    )

## 4. Feature engineering

`add_features` (in `pipeline.py`) builds new columns from each row on its
own: log transforms for skewed counts, the weekend share of the stay, month
of arrival and of booking placed on a circle, the weekday, and yes/no flags
for previous cancellations and stays. Because it learns nothing from the
data, it is safe to run before or after the split; it runs on each split
here so every step after the split is visibly applied to all three.

In [ ]:
train_df = add_features(train_df)
val_df = add_features(val_df)
test_df = add_features(test_df)

FEATURES = NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES
print(f"{len(FEATURES)} input columns before encoding")
print(train_df[NUMERIC_FEATURES].describe().round(2).T[["mean", "std", "min", "max"]])

## 5. Preprocessing (fitted on the training set only)

- Numeric columns: `StandardScaler` (mean 0, spread 1), so all features sit
  on a similar scale and gradient descent in PyTorch converges.
- Categorical columns: `OneHotEncoder`. Categories with fewer than 50
  training rows are merged into one "infrequent" group, and one column per
  feature is dropped so the one-hot columns don't duplicate the intercept.
  Categories never seen in training fall into the infrequent group.
- 0/1 flags pass through unchanged.

`fit_transform` on train learns the means, spreads and categories;
`transform` on validation and test only applies them, so no information
from later periods leaks into training.

In [ ]:
MIN_CATEGORY_ROWS = 50

preprocessor = ColumnTransformer(
    [
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ("binary", "passthrough", BINARY_FEATURES),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                min_frequency=MIN_CATEGORY_ROWS,
                handle_unknown="infrequent_if_exist",
                sparse_output=False,
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)

X_train = preprocessor.fit_transform(train_df[FEATURES])
X_val = preprocessor.transform(val_df[FEATURES])
X_test = preprocessor.transform(test_df[FEATURES])

y_train = train_df["y"].to_numpy()
y_val = val_df["y"].to_numpy()
y_test = test_df["y"].to_numpy()

feature_names = preprocessor.get_feature_names_out()

print("X_train:", X_train.shape, "  X_val:", X_val.shape, "  X_test:", X_test.shape)
print("Infrequent categories merged:")
for column, rare in zip(CATEGORICAL_FEATURES, preprocessor.named_transformers_["categorical"].infrequent_categories_):
    print(f"  {column}: {list(rare) if rare is not None else 'none'}")
print("\nFinal features:")
for name in feature_names:
    print(" ", name)